# Recipew Suggestion over Object Detection and Word Embeddings

# Google Colab Setup

If you're using Google Colab, first run this cell:

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Switch to the project directory (change according to your path)
%cd /content/drive/MyDrive/Intelligent_System

# Install dependencies from "requirements.txt"
!pip install -r requirements.txt

# Import dependencies

In [ ]:
from ultralytics import YOLO
import torch
import pandas as pd
import ast
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
import shutil
import yaml
import seaborn as sns
from gensim.models import Word2Vec, KeyedVectors

# Constants

In [ ]:
# Path of the folder containing files for the demo
demo_path = Path.cwd() / "demo_data"

CLASSES_SING = {
    0: 'apple',
    1: 'banana',
    2: 'beef',
    3: 'blueberry',
    4: 'bread',
    5: 'butter',
    6: 'carrot',
    7: 'cheese',
    8: 'chicken',
    9: 'chicken_breast',
    10: 'chocolate',
    11: 'corn',
    12: 'egg',
    13: 'flour',
    14: 'goat_cheese',
    15: 'green_bean',
    16: 'ground_beef',
    17: 'ham',
    18: 'heavy_cream',
    19: 'lime',
    20: 'milk',
    21: 'mushroom',
    22: 'onion',
    23: 'potato',
    24: 'shrimp',
    25: 'spinach',
    26: 'strawberry',
    27: 'sugar',
    28: 'sweet_potato',
    29: 'tomato'
}

# Initialization

Let's initialize everything required for Ingredient Detection:

In [ ]:
# instatiation of YOLO model
path_best_weights = demo_path / "weights.pt"
model = YOLO(str(path_best_weights))

Let's initialize everything required for Ingredient Substitution:



In [ ]:
# Load the embeddings
glove_embeddings = KeyedVectors.load(str(demo_path / "GloVe.model"))

Let's assume the user has a certain set of ingredients in their fridge. Using a custom-generated recipe dataset, we suggest a selection of recipes that can be prepared using the ingredients they have, sorted in ascending order by the number of ingredient substitutions required. Thus, the user will be presented with:
- First, recipes for which all the ingredients are already in the fridge;
- Next, recipes missing one ingredient, which can be substituted with similar ones available in the fridge;
- Then, recipes missing two ingredients, which can be substituted with similar ones available in the fridge;
- ...

For each missing ingredient, the possible substitutions are sorted in descending order of similarity to the original ingredient.

In [ ]:
def suggest_recipes(user_ingredients_set, recipes_for_suggestions_csv, embeddings, N = 50, max_recipes = 5, max_substitutions = 2):
    """
    Suggests recipes that can prepared using "user_ingredients_set", sorted in ascending order by the number of ingredient substitutions required
    (first 0 substitutions, then 1, then 2, etc.), up to a maximum of 'max_substitutions' missing ingredients.

    Parameters:
    - user_ingredients_set : a set containing the ingredients available to the user
    - recipes_for_suggestions_csv : path to the CSV file containing recipes that can be suggested to the user
    - embeddings : embeddings for each ingredient
    - N: the number of top most similar ingredients to consider for proposing a substitution
    - max_recipes : maximum number of recipes that can be suggested
    - max_substitutions : maximum number of missing ingredients that can be substituted

    Returns:
    - list of suggested recipes
    """

    # Read the recipes dataset
    df = pd.read_csv(recipes_for_suggestions_csv)

    # Initialize the list that will contain the suggested recipes
    all_valid_matches = []

    # Iterate through the recipes
    for index, row in df.iterrows():
        recipe_name = row['recipe']
        ingredients_raw = row['ingredients']

        # Convert the current recipe's ingredients list into a set
        recipe_ingredients_set = set(ast.literal_eval(ingredients_raw))

        # Identify missing ingredients from the current recipe by finding the difference between the recipe's ingredients set and user's ingredients set
        missing_ings = recipe_ingredients_set - user_ingredients_set

        # If more than "max_substitutions" ingredients are missing, discard the recipe
        if len(missing_ings) > max_substitutions:
            continue

        # If no ingredients are missing
        if not missing_ings:
            all_valid_matches.append({
                "Recipe_Name": recipe_name,
                "Original_Ingredients": recipe_ingredients_set,
                "Substitutions": {}  # 0 substitutions
            })
            continue

        # If at most 'max_substitutions' ingredients are missing, look for substitutions
        all_missing_covered = True
        missing_to_substitute = {}

        # For each missing ingredient
        for missing in missing_ings:
            is_covered = False

            if missing in embeddings.key_to_index:
                # Find the top-N most similar ingredients to the missing one
                top_n_similar = [word for word, score in embeddings.most_similar(missing, topn = N)]

                # Find which ingredients owned by the user appear in the top-N most similar keeping the ranking order, excluding those already in the current recipe
                available_subs = [word for word in top_n_similar if word in user_ingredients_set and word not in recipe_ingredients_set]

                if available_subs:
                    missing_to_substitute[missing] = available_subs
                    is_covered = True

            # If a missing ingredient has no substitutes among the user's available ones, discard the recipe
            if not is_covered:
                all_missing_covered = False
                break

        # If all missing ingredients have substitutes available to the user, suggest the recipe
        if all_missing_covered:
            all_valid_matches.append({
                "Recipe_Name": recipe_name,
                "Original_Ingredients": recipe_ingredients_set,
                "Substitutions": missing_to_substitute
            })

    # Sort the recipes in ascending order of necessary substitutions (first 0 substitutions, then 1, then 2, etc.)
    all_valid_matches.sort(key=lambda x: len(x["Substitutions"]))

    # Return the first 'max_recipes' recipes (the ones that require the less substitutions)
    return all_valid_matches[:max_recipes]

In [ ]:
# Path of the recipes dataset
recipes = str(demo_path / "recipes.csv")

# Perform Recipe Suggestion

In [ ]:
# Detect the ingredients in the input image
image_to_scan = demo_path / "input.jpg"
results = model(image_to_scan)
ingredients_list = list()

# Scan results
for result in results:
  ingredients_list = [CLASSES_SING[cls.item()] for cls in result.boxes.cls.int()]

# Remove duplicates
ingredients_set = set(ingredients_list)

# Suggest recipes that can be prepared with the detected ingredients, proposing substitutions in case of missing ingredients
recommendations = suggest_recipes(ingredients_set, recipes, glove_embeddings)

for rec in recommendations:
        print(f"- Recipe name: {rec['Recipe_Name']}")
        print(f"- Original ingredients: {', '.join(rec['Original_Ingredients'])}")

        if rec['Substitutions']:
            print("- Ingredients to substitute:")
            for missing_ing, substitutes in rec['Substitutions'].items():
                print(f"  * Substitute '{missing_ing}' with: {', '.join(substitutes)} (Available in the fridge)")
        else:
            print("- Ingredients to substitute: None (All ingredients are available in the fridge)")

        print("-" * 100)



image 1/1 /content/drive/.shortcut-targets-by-id/1fFJdTRbSGJGLoJB4meBgooeBmdBD4xbo/Intelligent_System/demo_data/input.jpg: 640x640 1 apple, 1 banana, 1 blueberries, 1 butter, 1 chicken, 1 corn, 1 flour, 1 green_beans, 1 ground_beef, 1 ham, 1 heavy_cream, 1 lime, 1 milk, 1 spinach, 1 sugar, 1758.5ms
Speed: 3.2ms preprocess, 1758.5ms inference, 5.2ms postprocess per image at shape (1, 3, 640, 640)
- Recipe name: Apple Blueberry Crumble
- Original ingredients: flour, apple, butter, blueberry
- Ingredients to substitute: None (All ingredients are available in the fridge)
----------------------------------------------------------------------------------------------------
- Recipe name: Lime Shrimp
- Original ingredients: lime, shrimp, butter
- Ingredients to substitute:
  * Substitute 'shrimp' with: chicken (Available in the fridge)
----------------------------------------------------------------------------------------------------
- Recipe name: Banana Bread
- Original ingredients: sugar,

# BACKUP (Do not execute the following cells)

In [ ]:
!pip install -U ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd ./drive/MyDrive/Intelligent_System

In [ ]:
INGREDIENT_DETECTION_FOLDER = Path.cwd() / "Ingredient_Detection"
INGREDIENT_SUBSTITUTION_FOLDER = Path.cwd() / "Ingredient_Detection"

In [ ]:
# instatiation of YOLO model
path_best_weights = INGREDIENT_DETECTION_FOLDER / "training" / "res" / "model_l" / "iter_3" / "weights" / "best.pt"
# pick an image at random in the dataset
image_to_scan = INGREDIENT_DETECTION_FOLDER / "datasets" / "refined" / "images" / "DSC_5677_JPG_jpg.rf.58174a6515f61e352358f1294d493edf.jpg"
model = YOLO(str(path_best_weights))
results = model(image_to_scan)
food = list()

#scan results
for result in results:
  food = [CLASSES_SING[cls.item()] for cls in result.boxes.cls.int()]

#delete the duplicates, in fact they're not adding any additional information
food = list(set(food))

print(food)

# print labels to verify correctness of prediction
labels = INGREDIENT_DETECTION_FOLDER / "datasets" / "refined" / "labels" / "DSC_5677_JPG_jpg.rf.58174a6515f61e352358f1294d493edf.txt"

with open(labels) as f :
  lbls = f.readlines()

  list_lbls = list()
  for l in lbls:
    app = l.split()[0]
    list_lbls.append(CLASSES_SING[int(app)])
  print(list_lbls)